In [6]:
from __future__ import annotations

import sys
from pathlib import Path


def _find_workspace_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root from the current working directory.")


WORKSPACE_DIR = _find_workspace_root(Path.cwd())
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
APPENDIX_DIR = CODE_DIR / "appendix"
APPENDIX_OUTPUTS_DIR = OUTPUTS_DIR / "appendix"
APPENDIX_RUNS_DIR = APPENDIX_OUTPUTS_DIR / "runs"
APPENDIX_FIGURES_DIR = APPENDIX_OUTPUTS_DIR / "figures"

for candidate in [CODE_DIR, APPENDIX_DIR]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

APPENDIX_RUNS_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)
print("appendix_outputs_root:", APPENDIX_OUTPUTS_DIR)


workspace_root: /Users/sallz/allzen/damadi/research/semantic-change/workspace
shared_assets_root: /Users/sallz/allzen/damadi/research/shared-assets
appendix_outputs_root: /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix


# Corpus-Native Broad Topic Network

This notebook builds a Gephi-ready co-occurrence network from a merged corpus source:
- the full ADS frame provides complete corpus coverage
- `dm_models.parquet` contributes stronger curated metadata such as `primary_arxiv_class`
- `abstract_clean` is used when available, with ADS `abstract` as fallback so the graph still covers the full corpus

The goal is a broad topical overview:
- candidate terms are extracted from the merged text field as `1` to `3` grams
- document-frequency thresholds scale with corpus size instead of imposing a hard top-`N` node cap
- edges connect terms that co-occur in the same abstract
- lexical containment edges such as `gamma -> gamma ray` are removed so the graph better reflects topic structure than phrase nesting
- node and edge tables carry `year`, `arxiv_class`, and `arxiv_category` summaries so Gephi can filter the graph by corpus metadata

This remains a broad abstract-level co-presence graph. It is intended as a map of topical neighborhoods in the corpus, not as a local syntactic or phrase-combination network.


In [7]:
import math
import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

from dm_term_normalization.tfidf_preproc import preproc, tokenize_preproc_text


In [8]:
NETWORK_ID = "broad_abstract_topic_network__merged_ads_dm_models_v1"
ADS_PATH = DATA_DIR / "parquet" / "ADS_MAIN_FRAME_2025.parquet"
DM_MODELS_PATH = DATA_DIR / "parquet" / "dm_models.parquet"

MAX_DOCS = None  # set to an integer like 5000 for a faster exploratory run
NGRAM_RANGE = (1, 3)

# These thresholds scale with corpus size so the graph stays broad but manageable.
MIN_TERM_DOC_SHARE = 0.003
MAX_TERM_DOC_SHARE = 0.12
MIN_EDGE_DOC_SHARE = 0.0015
MIN_EDGE_NPMI = 0.10

STOPWORDS_DIR = SHARED_ASSETS_DIR / "lists" / "tfidf"
SKLEARN_STOPWORDS_PATH = STOPWORDS_DIR / "sklearn_english_stopwords.txt"
FINAL_STOPWORDS_PATH = STOPWORDS_DIR / "final_stopwords.txt"

EXPORT_NETWORK_ID = NETWORK_ID if MAX_DOCS is None else f"{NETWORK_ID}__sample_{MAX_DOCS}"
EXPORT_DIR = APPENDIX_OUTPUTS_DIR / "gephi" / EXPORT_NETWORK_ID
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ads source:", ADS_PATH)
print("dm_models source:", DM_MODELS_PATH)
print("export dir:", EXPORT_DIR)
print("max docs:", MAX_DOCS if MAX_DOCS is not None else "full corpus")


ads source: /Users/sallz/allzen/damadi/research/semantic-change/workspace/data/parquet/ADS_MAIN_FRAME_2025.parquet
dm_models source: /Users/sallz/allzen/damadi/research/semantic-change/workspace/data/parquet/dm_models.parquet
export dir: /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1
max docs: full corpus


In [9]:
def load_wordlist(path: Path) -> set[str]:
    with open(path, encoding="utf-8") as f:
        return {
            line.strip()
            for line in f
            if line.strip() and not line.lstrip().startswith("#")
        }


STOPWORDS = load_wordlist(SKLEARN_STOPWORDS_PATH) | load_wordlist(FINAL_STOPWORDS_PATH)
STOPWORDS |= {"math", "token"}

GENERIC_TERMS = {
    "dark matter", "dark-matter", "lambda-cold", "lambda-cold dark-matter", "dark", "matter",
    "model", "models", "paper", "study", "studies", "data", "result", "results", "analysis",
    "approach", "method", "methods", "problem", "problems", "effect", "effects", "case",
    "cases", "work", "mass", "energy", "time", "space", "system", "systems", "universe",
    "parameter", "parameters", "constraint", "constraints", "observation", "observations",
    "evidence", "question", "possibility", "possible", "new", "recent", "based", "using",
    "however", "investigate", "consider", "compare", "compared", "produced", "future",
    "prediction", "predictions", "test", "tests", "state", "states", "value", "values",
    "set", "different", "physics", "properties", "property", "component", "components",
    "scenario", "scenarios", "effective", "total", "local", "times", "required", "relevant",
    "additional", "existing", "presence", "associated", "derived", "upper", "near",
    "naturally", "given", "obtain", "calculate", "examine", "explain", "suggest", "proposed",
    "lead", "leads", "include", "included", "production", "extended", "experimental", "viable",
}
WEAK_PARTS = {
    "new", "possible", "different", "various", "based", "using", "recent", "present", "show",
    "find", "study", "result", "however", "consider", "investigate", "compare", "compared",
    "produced", "future", "proposed", "given", "derived", "existing", "additional", "relevant",
    "viable",
}
BAD_PARTS = {
    "sub", "sup", "math", "token", "eq", "fig", "mml", "mrow", "mo", "mi", "mn",
    "msub", "msup", "msubsup",
}


def is_substantive(term: str) -> bool:
    tokens = term.split()
    if all(re.match(r"^[\d\.\-\+]+$", token) for token in tokens):
        return False
    if all(re.match(r"^\d{2,4}$", token) for token in tokens):
        return False
    return True


def is_valid_term(term: str) -> bool:
    if term in GENERIC_TERMS:
        return False
    if not is_substantive(term):
        return False

    parts = term.split()
    if any(part in BAD_PARTS for part in parts):
        return False

    if len(parts) == 1:
        token = parts[0]
        if token in STOPWORDS or len(token) < 4:
            return False
    else:
        if any(part in WEAK_PARTS for part in parts):
            return False
        if sum(part not in STOPWORDS for part in parts) < 2:
            return False
    return True


def safe_npmi(joint_count: np.ndarray, source_freq: np.ndarray, target_freq: np.ndarray, n_docs: int | np.ndarray) -> np.ndarray:
    p_xy = joint_count / n_docs
    p_x = source_freq / n_docs
    p_y = target_freq / n_docs
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log(p_xy / (p_x * p_y)) / -np.log(p_xy)
    out[~np.isfinite(out)] = np.nan
    return out


def term_contains(shorter: str, longer: str) -> bool:
    short_tokens = shorter.split()
    long_tokens = longer.split()
    if len(short_tokens) >= len(long_tokens):
        return False
    for idx in range(len(long_tokens) - len(short_tokens) + 1):
        if long_tokens[idx : idx + len(short_tokens)] == short_tokens:
            return True
    return False


def dominant_label(values: np.ndarray) -> tuple[str, float]:
    counts = Counter(values.tolist())
    label, n = max(counts.items(), key=lambda item: (item[1], item[0]))
    return label, n / len(values)


def summarize_doc_ids(doc_ids: np.ndarray, years: np.ndarray, arxiv_classes: np.ndarray, arxiv_categories: np.ndarray) -> dict[str, object]:
    if len(doc_ids) == 0:
        return {
            "year_min": pd.NA,
            "year_max": pd.NA,
            "year_mean": np.nan,
            "year_median": np.nan,
            "dominant_arxiv_class": pd.NA,
            "dominant_arxiv_class_share": np.nan,
            "dominant_arxiv_category": pd.NA,
            "dominant_arxiv_category_share": np.nan,
        }

    year_slice = years[doc_ids]
    class_slice = arxiv_classes[doc_ids]
    category_slice = arxiv_categories[doc_ids]
    dominant_class, dominant_class_share = dominant_label(class_slice)
    dominant_category, dominant_category_share = dominant_label(category_slice)

    return {
        "year_min": int(year_slice.min()),
        "year_max": int(year_slice.max()),
        "year_mean": float(year_slice.mean()),
        "year_median": float(np.median(year_slice)),
        "dominant_arxiv_class": dominant_class,
        "dominant_arxiv_class_share": float(dominant_class_share),
        "dominant_arxiv_category": dominant_category,
        "dominant_arxiv_category_share": float(dominant_category_share),
    }


In [10]:
ads = pd.read_parquet(
    ADS_PATH,
    columns=["bibcode", "year", "abstract", "arxiv_class", "arxiv_category"],
)
dm = pd.read_parquet(
    DM_MODELS_PATH,
    columns=["bibcode", "primary_arxiv_class", "arxiv_category", "abstract_clean"],
)

papers = ads.merge(dm, on="bibcode", how="left", suffixes=("_ads", "_dm"))
papers["network_text"] = papers["abstract_clean"]
papers["network_text"] = papers["network_text"].fillna("").astype(str)
needs_fallback = papers["network_text"].str.strip().eq("")
papers["text_source"] = np.where(needs_fallback, "ads_abstract", "dm_models_abstract_clean")
papers.loc[needs_fallback, "network_text"] = papers.loc[needs_fallback, "abstract"].fillna("").astype(str)

papers["merged_arxiv_class"] = papers["primary_arxiv_class"].fillna("").astype(str)
missing_class = papers["merged_arxiv_class"].str.strip().eq("")
papers.loc[missing_class, "merged_arxiv_class"] = papers.loc[missing_class, "arxiv_class"].fillna("MISSING").astype(str)
papers["merged_arxiv_class"] = papers["merged_arxiv_class"].replace("", "MISSING")

papers["merged_arxiv_category"] = papers["arxiv_category_dm"].fillna("").astype(str)
missing_category = papers["merged_arxiv_category"].str.strip().eq("")
papers.loc[missing_category, "merged_arxiv_category"] = papers.loc[missing_category, "arxiv_category_ads"].fillna("MISSING").astype(str)
papers["merged_arxiv_category"] = papers["merged_arxiv_category"].replace("", "MISSING")

papers["year"] = pd.to_numeric(papers["year"], errors="coerce")
papers = papers[papers["year"].notna()].copy()
papers["year"] = papers["year"].astype(int)
papers = papers[papers["network_text"].astype(str).str.strip().ne("")].copy()

if MAX_DOCS is not None:
    papers = papers.iloc[:MAX_DOCS].copy()

abstracts = papers["network_text"].tolist()
years = papers["year"].to_numpy(dtype=np.int32)
arxiv_classes = papers["merged_arxiv_class"].to_numpy(dtype=object)
arxiv_categories = papers["merged_arxiv_category"].to_numpy(dtype=object)

N = len(abstracts)
MIN_TERM_DOC_FREQ = max(20, math.ceil(N * MIN_TERM_DOC_SHARE))
MIN_EDGE_COUNT = max(10, math.ceil(N * MIN_EDGE_DOC_SHARE))
clean_source_count = int(papers["text_source"].eq("dm_models_abstract_clean").sum())
fallback_source_count = int(papers["text_source"].eq("ads_abstract").sum())
print(f"abstracts entering network build: {N:,}")
print("min term doc frequency:", MIN_TERM_DOC_FREQ)
print("max term doc frequency share:", MAX_TERM_DOC_SHARE)
print("min edge count:", MIN_EDGE_COUNT)
print("min edge npmi:", MIN_EDGE_NPMI)
print("clean abstracts used from dm_models:", clean_source_count)
print("fallback to ADS abstracts:", fallback_source_count)

vec = CountVectorizer(
    preprocessor=preproc,
    tokenizer=tokenize_preproc_text,
    token_pattern=None,
    lowercase=False,
    binary=True,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_TERM_DOC_FREQ,
    max_df=MAX_TERM_DOC_SHARE,
    stop_words=sorted(STOPWORDS),
)
X = vec.fit_transform(abstracts).astype(np.int32)
terms = vec.get_feature_names_out()
freq = np.asarray(X.sum(axis=0)).ravel()

keep_terms = np.array([is_valid_term(term) for term in terms])
X = X[:, keep_terms]
terms = terms[keep_terms]
freq = freq[keep_terms]

candidate_terms = pd.DataFrame(
    {
        "term": terms,
        "doc_freq": freq,
        "doc_share": freq / N,
        "n_tokens": [len(term.split()) for term in terms],
    }
).sort_values(["doc_freq", "term"], ascending=[False, True]).reset_index(drop=True)

print("candidate topical terms:", len(candidate_terms))
display(candidate_terms.head(30))


abstracts entering network build: 193,720
min term doc frequency: 582
max term doc frequency share: 0.12
min edge count: 291
min edge npmi: 0.1
clean abstracts used from dm_models: 173034
fallback to ADS abstracts: 20686
candidate topical terms: 3302


,term,doc_freq,doc_share,n_tokens
0,limit,23152,0.119513,1
1,survey,22765,0.117515,1
2,search,22662,0.116983,1
3,light,22555,0.116431,1
4,cluster,22271,0.114965,1
5,interaction,21341,0.110164,1
6,spectrum,20277,0.104672,1
7,galactic,20032,0.103407,1
8,massive,20010,0.103293,1
9,scalar,19085,0.098518,1


In [11]:
cooc = (X.T @ X).tocoo()
mask = cooc.row < cooc.col
rows = cooc.row[mask]
cols = cooc.col[mask]
counts = cooc.data[mask].astype(np.int32)
npmi = safe_npmi(counts.astype(float), freq[rows].astype(float), freq[cols].astype(float), N)

edges = pd.DataFrame(
    {
        "Source": terms[rows],
        "Target": terms[cols],
        "count": counts,
        "npmi": npmi,
        "source_doc_freq": freq[rows],
        "target_doc_freq": freq[cols],
        "source_idx": rows,
        "target_idx": cols,
    }
)
edges = edges[np.isfinite(edges["npmi"])].copy()
edges = edges[(edges["count"] >= MIN_EDGE_COUNT) & (edges["npmi"] >= MIN_EDGE_NPMI)].copy()

containment_mask = edges.apply(
    lambda row: term_contains(row["Source"], row["Target"]) or term_contains(row["Target"], row["Source"]),
    axis=1,
)
edges = edges[~containment_mask].copy().reset_index(drop=True)

X_csc = X.tocsc()
postings = {
    terms[idx]: X_csc.indices[X_csc.indptr[idx] : X_csc.indptr[idx + 1]]
    for idx in range(len(terms))
}

edge_meta_rows = []
for row in edges.itertuples(index=False):
    doc_ids = np.intersect1d(postings[row.Source], postings[row.Target], assume_unique=True)
    edge_meta_rows.append(summarize_doc_ids(doc_ids, years, arxiv_classes, arxiv_categories))

edges = pd.concat([edges, pd.DataFrame(edge_meta_rows)], axis=1)
edges["Weight"] = edges["count"]
edges = edges.drop(columns=["source_idx", "target_idx"])
edges = edges.sort_values(["count", "npmi"], ascending=[False, False]).reset_index(drop=True)

print("retained edges:", len(edges))
display(edges.head(30))


retained edges: 27021


,Source,Target,count,npmi,source_doc_freq,target_doc_freq,year_min,year_max,year_mean,year_median,dominant_arxiv_class,dominant_arxiv_class_share,dominant_arxiv_category,dominant_arxiv_category_share,Weight
0,power,spectrum,9923,0.597104,16078,20277,1984,2026,2014.200443,2016.0,astro-ph.CO,0.542276,astrophysics,0.796433,9923
1,detector,search,6782,0.362724,17186,22662,1985,2026,2015.450752,2017.0,MISSING,0.495429,Other,0.495429,6782
2,limit,search,5660,0.208624,23152,22662,1979,2026,2015.832862,2018.0,MISSING,0.311661,high-energy physics,0.434629,5660
3,cluster,x-ray,5301,0.419702,22271,10183,1972,2026,2010.394265,2011.0,astro-ph.CO,0.304094,astrophysics,0.725523,5301
4,cluster,survey,5163,0.187432,22271,22765,1981,2026,2013.559171,2015.0,astro-ph.CO,0.319582,astrophysics,0.766609,5163
5,decay,search,5095,0.248926,17608,22662,1984,2026,2016.821001,2018.0,hep-ph,0.412758,high-energy physics,0.579392,5095
6,search,signal,5086,0.230865,22662,18763,1985,2026,2017.359615,2019.0,MISSING,0.326190,high-energy physics,0.388911,5086
7,coupling,scalar,5045,0.303874,16901,19085,1985,2026,2017.547275,2019.0,hep-ph,0.464024,high-energy physics,0.549257,5045
8,cluster,massive,4997,0.212090,22271,20010,1926,2026,2013.671403,2015.0,astro-ph.GA,0.298979,astrophysics,0.757855,4997
9,dispersion,velocity,4952,0.557711,6564,18910,1942,2026,2011.424677,2013.0,astro-ph.GA,0.338045,astrophysics,0.770800,4952


In [12]:
node_ids = sorted(set(edges["Source"]) | set(edges["Target"]))
node_meta_rows = []
for term in node_ids:
    doc_ids = postings[term]
    row = {"Id": term, **summarize_doc_ids(doc_ids, years, arxiv_classes, arxiv_categories)}
    node_meta_rows.append(row)

node_meta = pd.DataFrame(node_meta_rows)
node_stats = candidate_terms.set_index("term").loc[node_ids].reset_index()
node_stats = node_stats.rename(columns={"term": "Id"})
node_stats["Label"] = node_stats["Id"]
node_stats["degree"] = (
    pd.concat(
        [edges[["Source"]].rename(columns={"Source": "Id"}), edges[["Target"]].rename(columns={"Target": "Id"})],
        ignore_index=True,
    )
    .value_counts("Id")
    .reindex(node_stats["Id"], fill_value=0)
    .to_numpy()
)
node_stats = node_stats.merge(node_meta, on="Id", how="left")
node_stats = node_stats[
    [
        "Id", "Label", "doc_freq", "doc_share", "degree", "n_tokens",
        "year_min", "year_max", "year_mean", "year_median",
        "dominant_arxiv_class", "dominant_arxiv_class_share",
        "dominant_arxiv_category", "dominant_arxiv_category_share",
    ]
].sort_values(["degree", "doc_freq", "Id"], ascending=[False, False, True]).reset_index(drop=True)

candidates_out = EXPORT_DIR / "candidate_terms.csv"
nodes_out = EXPORT_DIR / "nodes.csv"
edges_out = EXPORT_DIR / "edges.csv"
manifest_out = EXPORT_DIR / "manifest.json"

candidate_terms.to_csv(candidates_out, index=False)
node_stats.to_csv(nodes_out, index=False)
edges.to_csv(edges_out, index=False)

manifest = pd.Series(
    {
        "network_id": NETWORK_ID,
        "ads_path": str(ADS_PATH),
        "dm_models_path": str(DM_MODELS_PATH),
        "text_strategy": "abstract_clean_else_ads_abstract",
        "arxiv_class_strategy": "primary_arxiv_class_else_ads_arxiv_class",
        "arxiv_category_strategy": "dm_arxiv_category_else_ads_arxiv_category",
        "n_abstracts": int(N),
        "ngram_range": list(NGRAM_RANGE),
        "max_docs": None if MAX_DOCS is None else int(MAX_DOCS),
        "min_term_doc_share": float(MIN_TERM_DOC_SHARE),
        "max_term_doc_share": float(MAX_TERM_DOC_SHARE),
        "min_term_doc_freq": int(MIN_TERM_DOC_FREQ),
        "min_edge_doc_share": float(MIN_EDGE_DOC_SHARE),
        "min_edge_count": int(MIN_EDGE_COUNT),
        "min_edge_npmi": float(MIN_EDGE_NPMI),
        "candidate_terms": int(len(candidate_terms)),
        "retained_nodes": int(len(node_stats)),
        "retained_edges": int(len(edges)),
    }
)
manifest.to_json(manifest_out, indent=2)

print("wrote:")
print("  candidate terms ->", candidates_out)
print("  nodes ->", nodes_out)
print("  edges ->", edges_out)
print("  manifest ->", manifest_out)

display(node_stats.head(30))


wrote:
  candidate terms -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1/candidate_terms.csv
  nodes -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1/nodes.csv
  edges -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1/edges.csv
  manifest -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1/manifest.json


,Id,Label,doc_freq,doc_share,degree,n_tokens,year_min,year_max,year_mean,year_median,dominant_arxiv_class,dominant_arxiv_class_share,dominant_arxiv_category,dominant_arxiv_category_share
0,survey,survey,22765,0.117515,498,1,1959,2026,2015.200264,2017.0,astro-ph.CO,0.331342,astrophysics,0.768592
1,stellar,stellar,17664,0.091183,463,1,1926,2026,2014.572973,2016.0,astro-ph.GA,0.404721,astrophysics,0.728997
2,search,search,22662,0.116983,384,1,1979,2026,2015.618922,2017.0,MISSING,0.349837,high-energy physics,0.354867
3,cluster,cluster,22271,0.114965,383,1,1926,2026,2011.814602,2013.0,MISSING,0.261596,astrophysics,0.703381
4,detector,detector,17186,0.088716,341,1,1984,2026,2015.236937,2017.0,MISSING,0.437333,Other,0.437333
5,population,population,11590,0.059829,335,1,1972,2026,2014.299396,2016.0,astro-ph.GA,0.331924,astrophysics,0.769629
6,velocity,velocity,18910,0.097615,316,1,1926,2026,2012.532628,2014.0,astro-ph.GA,0.267002,astrophysics,0.675410
7,research,research,8039,0.041498,298,1,1961,2026,2015.940664,2018.0,MISSING,0.591243,Other,0.591243
8,science,science,7346,0.037921,288,1,1972,2026,2015.887013,2017.0,MISSING,0.534304,Other,0.534304
9,galactic,galactic,20032,0.103407,280,1,1926,2026,2013.162740,2015.0,astro-ph.GA,0.283247,astrophysics,0.662939


## Long-Format Edge Events

This companion export keeps one row per retained document-level co-occurrence instead of collapsing everything to a single edge summary.

- repeated `Source` and `Target` pairs preserve exact `bibcode`, `year`, `arxiv_class`, and `arxiv_category` values
- aggregate edge statistics such as `edge_count` and `edge_npmi` are repeated on each row so the long table can still be reconciled with the broad network
- the files are written to a sibling folder next to the broad network export so you can compare both formats before deciding whether the detailed version is worth keeping in your main workflow

The primary output is Parquet because the full-corpus event table can become very large. You can toggle raw CSV export on in the code cell below if you want a direct flat file for Gephi import experiments.


In [14]:
import pyarrow as pa
import pyarrow.parquet as pq

LONG_FORMAT_NETWORK_ID = f"{NETWORK_ID}__event_level"
LONG_FORMAT_EXPORT_ID = LONG_FORMAT_NETWORK_ID if MAX_DOCS is None else f"{LONG_FORMAT_NETWORK_ID}__sample_{MAX_DOCS}"
LONG_FORMAT_EXPORT_DIR = APPENDIX_OUTPUTS_DIR / "gephi" / LONG_FORMAT_EXPORT_ID
LONG_FORMAT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_LONG_FORMAT_CSV = True  # switch on if you want a raw CSV alongside the Parquet export
LONG_FORMAT_BATCH_ROWS = 250_000

long_edges_out = LONG_FORMAT_EXPORT_DIR / "edge_events.parquet"
long_edges_csv_out = LONG_FORMAT_EXPORT_DIR / "edge_events.csv"
long_manifest_out = LONG_FORMAT_EXPORT_DIR / "manifest.json"

paper_bibcodes = papers["bibcode"].fillna("MISSING").astype(str).to_numpy(dtype=object)
paper_years = papers["year"].to_numpy(dtype=np.int32)
paper_arxiv_classes = papers["merged_arxiv_class"].fillna("MISSING").astype(str).to_numpy(dtype=object)
paper_arxiv_categories = papers["merged_arxiv_category"].fillna("MISSING").astype(str).to_numpy(dtype=object)
paper_text_sources = papers["text_source"].fillna("MISSING").astype(str).to_numpy(dtype=object)

long_event_schema = pa.schema(
    [
        ("Source", pa.string()),
        ("Target", pa.string()),
        ("bibcode", pa.string()),
        ("year", pa.int32()),
        ("arxiv_class", pa.string()),
        ("arxiv_category", pa.string()),
        ("text_source", pa.string()),
        ("edge_count", pa.int32()),
        ("edge_npmi", pa.float64()),
        ("source_doc_freq", pa.int32()),
        ("target_doc_freq", pa.int32()),
    ]
)

if long_edges_out.exists():
    long_edges_out.unlink()
if WRITE_LONG_FORMAT_CSV and long_edges_csv_out.exists():
    long_edges_csv_out.unlink()

writer = None
batch_frames: list[pd.DataFrame] = []
batch_rows = 0
total_event_rows = 0
preview = pd.DataFrame()

for edge_row in edges.itertuples(index=False):
    doc_ids = np.intersect1d(postings[edge_row.Source], postings[edge_row.Target], assume_unique=True)
    if len(doc_ids) == 0:
        continue

    batch = pd.DataFrame(
        {
            "Source": np.repeat(edge_row.Source, len(doc_ids)),
            "Target": np.repeat(edge_row.Target, len(doc_ids)),
            "bibcode": paper_bibcodes[doc_ids],
            "year": paper_years[doc_ids],
            "arxiv_class": paper_arxiv_classes[doc_ids],
            "arxiv_category": paper_arxiv_categories[doc_ids],
            "text_source": paper_text_sources[doc_ids],
            "edge_count": np.repeat(int(edge_row.count), len(doc_ids)),
            "edge_npmi": np.repeat(float(edge_row.npmi), len(doc_ids)),
            "source_doc_freq": np.repeat(int(edge_row.source_doc_freq), len(doc_ids)),
            "target_doc_freq": np.repeat(int(edge_row.target_doc_freq), len(doc_ids)),
        }
    )

    if preview.empty:
        preview = batch.head(20).copy()

    batch_frames.append(batch)
    batch_rows += len(batch)
    total_event_rows += len(batch)

    if batch_rows >= LONG_FORMAT_BATCH_ROWS:
        chunk = pd.concat(batch_frames, ignore_index=True)
        table = pa.Table.from_pandas(chunk, schema=long_event_schema, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(long_edges_out, long_event_schema)
        writer.write_table(table)
        if WRITE_LONG_FORMAT_CSV:
            chunk.to_csv(
                long_edges_csv_out,
                index=False,
                mode="a",
                header=not long_edges_csv_out.exists(),
            )
        batch_frames = []
        batch_rows = 0

if batch_frames:
    chunk = pd.concat(batch_frames, ignore_index=True)
    table = pa.Table.from_pandas(chunk, schema=long_event_schema, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(long_edges_out, long_event_schema)
    writer.write_table(table)
    if WRITE_LONG_FORMAT_CSV:
        chunk.to_csv(
            long_edges_csv_out,
            index=False,
            mode="a",
            header=not long_edges_csv_out.exists(),
        )

if writer is not None:
    writer.close()
else:
    empty_chunk = pd.DataFrame(
        {
            "Source": pd.Series(dtype="string"),
            "Target": pd.Series(dtype="string"),
            "bibcode": pd.Series(dtype="string"),
            "year": pd.Series(dtype="int32"),
            "arxiv_class": pd.Series(dtype="string"),
            "arxiv_category": pd.Series(dtype="string"),
            "text_source": pd.Series(dtype="string"),
            "edge_count": pd.Series(dtype="int32"),
            "edge_npmi": pd.Series(dtype="float64"),
            "source_doc_freq": pd.Series(dtype="int32"),
            "target_doc_freq": pd.Series(dtype="int32"),
        }
    )
    pq.write_table(pa.Table.from_pandas(empty_chunk, schema=long_event_schema, preserve_index=False), long_edges_out)

long_manifest = pd.Series(
    {
        "network_id": LONG_FORMAT_NETWORK_ID,
        "base_network_id": NETWORK_ID,
        "n_abstracts": int(N),
        "retained_edge_pairs": int(len(edges)),
        "event_rows": int(total_event_rows),
        "max_docs": None if MAX_DOCS is None else int(MAX_DOCS),
        "write_csv": bool(WRITE_LONG_FORMAT_CSV),
        "parquet_path": str(long_edges_out),
        "csv_path": str(long_edges_csv_out) if WRITE_LONG_FORMAT_CSV else None,
        "text_strategy": "abstract_clean_else_ads_abstract",
        "arxiv_class_strategy": "primary_arxiv_class_else_ads_arxiv_class",
        "arxiv_category_strategy": "dm_arxiv_category_else_ads_arxiv_category",
        "min_term_doc_share": float(MIN_TERM_DOC_SHARE),
        "max_term_doc_share": float(MAX_TERM_DOC_SHARE),
        "min_term_doc_freq": int(MIN_TERM_DOC_FREQ),
        "min_edge_doc_share": float(MIN_EDGE_DOC_SHARE),
        "min_edge_count": int(MIN_EDGE_COUNT),
        "min_edge_npmi": float(MIN_EDGE_NPMI),
    }
)
long_manifest.to_json(long_manifest_out, indent=2)

print("wrote long-format edge events:")
print("  export dir ->", LONG_FORMAT_EXPORT_DIR)
print("  parquet ->", long_edges_out)
if WRITE_LONG_FORMAT_CSV:
    print("  csv ->", long_edges_csv_out)
print("  manifest ->", long_manifest_out)
print("  event rows ->", total_event_rows)

display(preview)


wrote long-format edge events:
  export dir -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1__event_level
  parquet -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1__event_level/edge_events.parquet
  csv -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1__event_level/edge_events.csv
  manifest -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/broad_abstract_topic_network__merged_ads_dm_models_v1__event_level/manifest.json
  event rows -> 15364516


,Source,Target,bibcode,year,arxiv_class,arxiv_category,text_source,edge_count,edge_npmi,source_doc_freq,target_doc_freq
0,power,spectrum,2004NuPhB.683..219B,2004,hep-ph,high-energy physics,dm_models_abstract_clean,9923,0.597104,16078,20277
1,power,spectrum,2013MNRAS.428.1774B,2013,astro-ph.CO,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
2,power,spectrum,2002PhRvD..66h3505B,2002,astro-ph,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
3,power,spectrum,2004PhRvD..69l3524S,2004,astro-ph,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
4,power,spectrum,2025A&A...697A.213D,2025,astro-ph.CO,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
5,power,spectrum,1999ApJ...511....5E,1999,astro-ph,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
6,power,spectrum,2001MNRAS.321..372J,2001,astro-ph,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
7,power,spectrum,1984Natur.311..517B,1984,MISSING,Other,dm_models_abstract_clean,9923,0.597104,16078,20277
8,power,spectrum,2021MNRAS.506.4210I,2021,astro-ph.CO,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277
9,power,spectrum,2024MNRAS.527.2835S,2024,astro-ph.GA,astrophysics,dm_models_abstract_clean,9923,0.597104,16078,20277


## Edge Slices

This export keeps one row per retained term pair within each metadata slice rather than one row per paper.

- slice key: `year + arxiv_class + arxiv_category`
- each row remains Gephi- and Cytoscape-friendly as an edge table with edge attributes
- this is intended as a middle layer between the broad aggregate graph and the full document-level event table
- unlike the dominant-class summaries in the aggregate graph, these rows preserve the actual slice membership directly


In [15]:
from scipy import sparse

SLICE_EXPORT_ID = "edge_slices" if MAX_DOCS is None else f"edge_slices__sample_{MAX_DOCS}"
SLICE_EXPORT_DIR = APPENDIX_OUTPUTS_DIR / "gephi" / SLICE_EXPORT_ID
SLICE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SLICE_BATCH_ROWS = 100_000
slice_edges_out = SLICE_EXPORT_DIR / "edges_by_slice.csv"
slice_manifest_out = SLICE_EXPORT_DIR / "manifest.json"

if slice_edges_out.exists():
    slice_edges_out.unlink()

slice_doc_meta = pd.DataFrame(
    {
        "year": years,
        "arxiv_class": arxiv_classes,
        "arxiv_category": arxiv_categories,
    }
)
slice_index = pd.MultiIndex.from_frame(slice_doc_meta)
slice_codes, slice_labels = pd.factorize(slice_index)
slice_lookup = slice_labels.to_frame(index=False)
slice_lookup.columns = ["year", "arxiv_class", "arxiv_category"]
slice_lookup["docs_in_slice"] = np.bincount(slice_codes, minlength=len(slice_lookup)).astype(np.int32)

slice_years = slice_lookup["year"].to_numpy(dtype=np.int32)
slice_classes = slice_lookup["arxiv_class"].to_numpy(dtype=object)
slice_categories = slice_lookup["arxiv_category"].to_numpy(dtype=object)
slice_sizes = slice_lookup["docs_in_slice"].to_numpy(dtype=np.int32)

term_to_idx = {term: idx for idx, term in enumerate(terms)}
retained_terms = node_ids
retained_term_indices = np.array([term_to_idx[term] for term in retained_terms], dtype=np.int32)
retained_term_lookup = {term: idx for idx, term in enumerate(retained_terms)}

doc_index = np.arange(N, dtype=np.int32)
slice_indicator = sparse.csr_matrix(
    (np.ones(N, dtype=np.int8), (doc_index, slice_codes)),
    shape=(N, len(slice_lookup)),
)
term_slice_counts = (X[:, retained_term_indices].T @ slice_indicator).tocsr()

slice_frames: list[pd.DataFrame] = []
slice_batch_rows = 0
total_slice_rows = 0
slice_preview = pd.DataFrame()

for edge_row in edges.itertuples(index=False):
    doc_ids = np.intersect1d(postings[edge_row.Source], postings[edge_row.Target], assume_unique=True)
    if len(doc_ids) == 0:
        continue

    edge_slice_counts = np.bincount(slice_codes[doc_ids], minlength=len(slice_lookup)).astype(np.int32)
    active_slice_codes = np.flatnonzero(edge_slice_counts)
    if len(active_slice_codes) == 0:
        continue

    source_slice_freq = np.asarray(term_slice_counts[retained_term_lookup[edge_row.Source], active_slice_codes].toarray()).ravel().astype(np.int32)
    target_slice_freq = np.asarray(term_slice_counts[retained_term_lookup[edge_row.Target], active_slice_codes].toarray()).ravel().astype(np.int32)
    counts_in_slice = edge_slice_counts[active_slice_codes]
    docs_in_slice = slice_sizes[active_slice_codes]
    slice_npmi = safe_npmi(
        counts_in_slice.astype(float),
        source_slice_freq.astype(float),
        target_slice_freq.astype(float),
        docs_in_slice.astype(float),
    )

    batch = pd.DataFrame(
        {
            "Source": np.repeat(edge_row.Source, len(active_slice_codes)),
            "Target": np.repeat(edge_row.Target, len(active_slice_codes)),
            "year": slice_years[active_slice_codes],
            "arxiv_class": slice_classes[active_slice_codes],
            "arxiv_category": slice_categories[active_slice_codes],
            "count": counts_in_slice,
            "npmi": slice_npmi,
            "docs_in_slice": docs_in_slice,
            "edge_doc_share": counts_in_slice / docs_in_slice,
            "source_slice_doc_freq": source_slice_freq,
            "target_slice_doc_freq": target_slice_freq,
            "global_count": np.repeat(int(edge_row.count), len(active_slice_codes)),
            "global_npmi": np.repeat(float(edge_row.npmi), len(active_slice_codes)),
        }
    )
    batch = batch[np.isfinite(batch["npmi"])].copy()
    if batch.empty:
        continue

    if slice_preview.empty:
        slice_preview = batch.head(20).copy()

    slice_frames.append(batch)
    slice_batch_rows += len(batch)
    total_slice_rows += len(batch)

    if slice_batch_rows >= SLICE_BATCH_ROWS:
        chunk = pd.concat(slice_frames, ignore_index=True)
        chunk.to_csv(
            slice_edges_out,
            index=False,
            mode="a",
            header=not slice_edges_out.exists(),
        )
        slice_frames = []
        slice_batch_rows = 0

if slice_frames:
    chunk = pd.concat(slice_frames, ignore_index=True)
    chunk.to_csv(
        slice_edges_out,
        index=False,
        mode="a",
        header=not slice_edges_out.exists(),
    )
elif not slice_edges_out.exists():
    pd.DataFrame(
        columns=[
            "Source",
            "Target",
            "year",
            "arxiv_class",
            "arxiv_category",
            "count",
            "npmi",
            "docs_in_slice",
            "edge_doc_share",
            "source_slice_doc_freq",
            "target_slice_doc_freq",
            "global_count",
            "global_npmi",
        ]
    ).to_csv(slice_edges_out, index=False)

slice_manifest = pd.Series(
    {
        "export_id": SLICE_EXPORT_ID,
        "n_abstracts": int(N),
        "retained_edge_pairs": int(len(edges)),
        "slice_rows": int(total_slice_rows),
        "slice_key": ["year", "arxiv_class", "arxiv_category"],
        "max_docs": None if MAX_DOCS is None else int(MAX_DOCS),
        "csv_path": str(slice_edges_out),
        "min_term_doc_share": float(MIN_TERM_DOC_SHARE),
        "max_term_doc_share": float(MAX_TERM_DOC_SHARE),
        "min_term_doc_freq": int(MIN_TERM_DOC_FREQ),
        "min_edge_doc_share": float(MIN_EDGE_DOC_SHARE),
        "min_edge_count": int(MIN_EDGE_COUNT),
        "min_edge_npmi": float(MIN_EDGE_NPMI),
    }
)
slice_manifest.to_json(slice_manifest_out, indent=2)

print("wrote edge slices:")
print("  export dir ->", SLICE_EXPORT_DIR)
print("  csv ->", slice_edges_out)
print("  manifest ->", slice_manifest_out)
print("  slice rows ->", total_slice_rows)

display(slice_preview)


/var/folders/0y/s2gghpcj4zj9wp6w8r7nkmq00000gq/T/ipykernel_63202/909390293.py:75: RuntimeWarning: invalid value encountered in divide
  out = np.log(p_xy / (p_x * p_y)) / -np.log(p_xy)


wrote edge slices:
  export dir -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/edge_slices
  csv -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/edge_slices/edges_by_slice.csv
  manifest -> /Users/sallz/allzen/damadi/research/semantic-change/workspace/outputs/appendix/gephi/edge_slices/manifest.json
  slice rows -> 2736745


,Source,Target,year,arxiv_class,arxiv_category,count,npmi,docs_in_slice,edge_doc_share,source_slice_doc_freq,target_slice_doc_freq,global_count,global_npmi
0,power,spectrum,2015,hep-ph,high-energy physics,8,0.320556,1377,0.005810,15,141,9923,0.597104
1,power,spectrum,2011,astro-ph.CO,astrophysics,228,0.769010,2000,0.114000,291,295,9923,0.597104
2,power,spectrum,2009,hep-ph,high-energy physics,1,0.109715,837,0.001195,4,100,9923,0.597104
3,power,spectrum,2019,astro-ph.GA,astrophysics,19,0.417478,1577,0.012048,74,64,9923,0.597104
4,power,spectrum,1994,hep-ph,high-energy physics,1,0.627662,123,0.008130,1,6,9923,0.597104
5,power,spectrum,2009,astro-ph.CO,astrophysics,104,0.650225,1177,0.088360,156,162,9923,0.597104
6,power,spectrum,2006,hep-ph,high-energy physics,4,0.354887,499,0.008016,6,60,9923,0.597104
7,power,spectrum,2003,astro-ph,astrophysics,170,0.766180,1609,0.105656,208,235,9923,0.597104
8,power,spectrum,2001,astro-ph,astrophysics,157,0.693528,1423,0.110330,207,234,9923,0.597104
9,power,spectrum,2001,hep-ph,high-energy physics,2,0.262550,349,0.005731,4,45,9923,0.597104
